# Gao validation: positive-class weighting

This development experiment compares PCBert-Kla and token-gated MM-Kla with unweighted BCE and with positive examples weighted by the Gao training-set negative/positive ratio. All four runs use the same locked Gao train/validation split, ProtBert, SGD, seed, and checkpoint-selection rule. No Gao internal-test or rice predictions are generated. The previously inspected test results cannot be treated as an untouched blind evaluation of later changes.

Kaggle settings: enable Internet and one GPU (T4 x2 is also suitable). Provide a fine-grained, read-only GitHub token through the Kaggle secret `GITHUB_TOKEN`; the token is not printed or stored in output files.

## 1. Repository and environment

The repository commit and package versions identify the code and environment used for the comparison.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'MM-Kla'
PROJECT_DIR = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT_ROOT = PROJECT_DIR / 'outputs' / 'kaggle_weighted_development'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
if not WORKING_DIR.is_dir():
    raise RuntimeError('A Kaggle runtime is required.')
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing.')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" x-access-token ;;\n'
            '  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n'
            '  *) exit 1 ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0',
                    'MM_KLA_GITHUB_TOKEN': token})
        try:
            clone = subprocess.run(
                ['git', '-c', 'credential.helper=', 'clone', '--depth', '1',
                 '--branch', 'main', REPO_URL, str(REPO_DIR)],
                env=env, capture_output=True, text=True, check=False,
            )
        finally:
            env.pop('MM_KLA_GITHUB_TOKEN', None)
            del token
    if clone.returncode:
        raise RuntimeError('Private clone failed; check Internet and secret access.')
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} is not a Git checkout.')
if not (PROJECT_DIR / 'scripts/run_replication.py').is_file():
    raise RuntimeError('The source-study runner is missing from the checkout.')
COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR,
    capture_output=True, text=True, check=True,
).stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
               cwd=PROJECT_DIR, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle settings.')
LOCK_PATH = PROJECT_DIR / 'review_data/locked_source_study_v1/protocol_lock.json'
LOCK_SHA256 = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
subprocess.run([sys.executable, 'scripts/run_replication.py',
                '--run', 'source-study-check'], cwd=PROJECT_DIR, check=True)
runtime = {
    'git_commit': COMMIT, 'source_lock_sha256': LOCK_SHA256,
    'python': sys.version.split()[0],
    'gpu': torch.cuda.get_device_name(0),
    'packages': {name: importlib.metadata.version(name) for name in
                 ('numpy', 'pandas', 'scikit-learn', 'torch', 'transformers')},
}
(WORKING_DIR / 'mm_kla_weighted_runtime.json').write_text(
    json.dumps(runtime, indent=2, sort_keys=True) + '\n'
)
print(json.dumps(runtime, indent=2, sort_keys=True))

## 2. Matched development runs

Each model uses four trainable ProtBert encoder layers, batch size 4, SGD at 0.003, 30 epochs, and seed 42. The checkpoint is selected by Gao validation AUPRC. Weighting is calculated from the Gao training partition only. Validation loss remains unweighted so the two settings are directly interpretable. Runs are sequential on `cuda:0`; no test-set inference occurs.

In [ ]:
ARCHITECTURES = ('baseline', 'token_gated')
WEIGHTINGS = ('none', 'train_ratio')
SEED = 42
RUN_DIRS = {}
for architecture in ARCHITECTURES:
    for weighting in WEIGHTINGS:
        command = [
            sys.executable, 'scripts/run_replication.py',
            '--run', 'source-study-development',
            '--architecture', architecture,
            '--positive-weighting', weighting,
            '--model-name', 'Rostlab/prot_bert',
            '--encoder-layers', '4', '--epochs', '30',
            '--batch-size', '4', '--device', 'cuda:0',
            '--optimizer', 'sgd', '--learning-rate', '0.003',
            '--weight-decay', '0', '--scheduler', 'none',
            '--seed', str(SEED), '--output-dir', str(OUTPUT_ROOT),
            '--save-models',
        ]
        print('Running:', architecture, weighting, flush=True)
        subprocess.run(command, cwd=PROJECT_DIR, check=True)
        root = OUTPUT_ROOT / 'source_study' / 'runs'
        matches = []
        for config_path in root.glob('*/run_config.json'):
            config = json.loads(config_path.read_text())
            if (config.get('run') == 'source-study-development'
                    and config.get('architecture') == architecture
                    and config.get('positive_weighting') == weighting
                    and config.get('model_seed') == SEED
                    and config.get('source_lock_sha256') == LOCK_SHA256
                    and config.get('training_rows') == 3599
                    and config.get('validation_rows') == 519):
                matches.append(config_path.parent)
        if len(matches) != 1 or not (matches[0] / 'validation_metrics.json').is_file():
            raise RuntimeError(f'Completed run not uniquely identified: {architecture}, {weighting}')
        RUN_DIRS[(architecture, weighting)] = matches[0]
        print('Saved run:', matches[0], flush=True)

## 3. Gao validation comparison

The table reads metrics and class counts from the saved run artifacts; no metric is entered manually. This is a single-seed development comparison, not evidence of generalization or a claim of improvement on unseen proteins. AUPRC is the primary selection metric.

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
metrics_to_show = ('AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC')
for (architecture, weighting), run_dir in RUN_DIRS.items():
    config = json.loads((run_dir / 'run_config.json').read_text())
    metrics = json.loads((run_dir / 'validation_metrics.json').read_text())
    if config['test_rows'] != 0 or config['extra_test_rows']:
        raise RuntimeError('A development run unexpectedly evaluated a test set.')
    rows.append({
        'architecture': architecture, 'positive_weighting': weighting,
        'seed': config['model_seed'], 'selected_epoch': config['selected_epoch'],
        'train_positive': config['training_positive_rows'],
        'train_negative': config['training_negative_rows'],
        'positive_weight': config['training_positive_weight'],
        **{name: metrics[name] for name in metrics_to_show},
    })
results = pd.DataFrame(rows)
display(results.round(4))
auprc = results.pivot(index='architecture', columns='positive_weighting', values='AUPRC')
auprc['weighted_minus_unweighted'] = auprc['train_ratio'] - auprc['none']
display(auprc.round(4))
summary_path = WORKING_DIR / 'mm_kla_weighted_validation.csv'
results.to_csv(summary_path, index=False)
print('Saved validation summary:', summary_path)
print('Saved run artifacts:', OUTPUT_ROOT / 'source_study' / 'runs')